In [ ]:
import os
import scanpy as sc
import pandas as pd
import scanpy.external as sce

# Preprocess

In [ ]:
folder_path = '../3_rna_atac_cell/data_rna/'
mtx_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.mtx')]

In [ ]:
adata_list = [sc.read_mtx(mtx_file) for mtx_file in mtx_files]

In [ ]:
folder_path = '../3_rna_atac_cell/data_rna/'
csv_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.startswith('cell')]

In [ ]:
cell_list = [pd.read_csv(csv_file, index_col=0) for csv_file in csv_files]

In [ ]:
adata_list

In [ ]:
for i in range(len(adata_list)):
    adata_list[i].var_names = cell_list[i]['cell_id'].tolist()
    adata_list[i] = adata_list[i].T

In [ ]:
combined = adata_list[0].concatenate(adata_list[1], index_unique=None)

In [ ]:
combined = combined.concatenate(adata_list[2], index_unique=None)

In [ ]:
gene = pd.read_csv('../3_rna_atac_cell/data_rna/gene.csv', index_col=0)
gene

In [ ]:
combined.var_names = gene['gene'].tolist()

In [ ]:
del combined.obs['batch']
combined

In [ ]:
df = pd.DataFrame([x.split('-') for x in combined.obs_names], columns=['part1', 'part2', 'part3'])

In [ ]:
df

In [ ]:
combined.obs['dataset'] = df['part1'].tolist()
combined.obs['barcode'] = df['part2'].tolist()

In [ ]:
df['part4'] = df['part1'] + '-' + df['part3']

In [ ]:
df

In [ ]:
combined.obs['sample'] = df['part4'].tolist()

In [ ]:
combined.write('rna.h5ad', compression='gzip')

In [ ]:
adata = sc.read_h5ad('./rna.h5ad')
adata

In [ ]:
ann = pd.read_csv('../4_cell_ann/ann_merge.csv', index_col=0)
ann

In [ ]:
df = pd.DataFrame([x.split('-') for x in ann.index], columns=['part1', 'part2', 'part3'])
df['part4'] = df['part1'] + '-' + df['part3']

In [ ]:
ann['dataset'] = df['part1'].tolist()
ann['sample'] = df['part4'].tolist()

In [ ]:
ann

In [ ]:
ann_filter = ann.dropna(subset=['cellType']) # Remove missing values
ann_filter

In [ ]:
test = ann_filter[ann_filter['cellType']=='Proerythroblast']
test

In [ ]:
test['cellType'].value_counts()

In [ ]:
ann_all = pd.DataFrame(adata.obs['sample'].value_counts())
ann_all

In [ ]:
ann_all.to_csv('./processed/original_ann/sample_all.csv')

# Try harmony

In [ ]:
adata = sc.read_h5ad('./processed/all_cell/rna_leiden.h5ad')
adata

In [ ]:
# sc.pp.calculate_qc_metrics(adata,inplace=True)
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
# adata.raw = adata
# adata = adata[:, adata.var.highly_variable]
# sc.pp.scale(adata, max_value=10)
# sc.tl.pca(adata, svd_solver='arpack')
# sc.pl.pca_variance_ratio(adata, log=True)

# sce.pp.harmony_integrate(adata, 'sample')
# adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']
# sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# adata.write('rna_neighbor.h5ad', compression='gzip')

# sc.tl.umap(adata)
# sc.tl.leiden(adata,resolution=1)

# adata.write('rna_leiden.h5ad', compression='gzip')

In [ ]:
# sc.pl.umap(adata, color='dataset')

In [ ]:
sc.tl.leiden(adata,resolution=0.05)

In [ ]:
# sc.pl.umap(adata, color='leiden', save='umap')

In [ ]:
# sc.pl.dotplot(adata, 
#               groupby='leiden', 
#               var_names=[
#                          # 'LEPR','CXCL12','NCAM1', # Mesenchymal
#                          # 'TAGLN','ACTA2', # Vascular smooth muscle
#                          # 'EMCN', 'FLT1', # Endothelial cell
#                          'S100A9','LYZ','SRGN',# Myeloid
#                          'MZB1','GZMA','IL7R','MPO','DNTT','CD79B','PAX5', # Lymphoid
#                          'ITGA2B','HBA1','HBA2','GATA1', # Megakaryocyte/Erythroid
#                          'AVP','SPINK2' # HSPC
#                         ])

In [ ]:
adata.obs['leiden'].value_counts()

In [ ]:
adata_1 = adata[adata.obs['leiden'] == '1']

In [ ]:
adata_1

In [ ]:
adata.obs['dataset'].value_counts()

In [ ]:
adata_1.obs['dataset'].value_counts()

# Filter annotated cells using ann_filter

## Preprocess for ann_merge

In [ ]:
ann = pd.read_csv('../4_cell_ann/ann_merge.csv', index_col=0)
ann

In [ ]:
exclude_values = ['Doublet','Doublet.B','Doublet.PB','Doublet1','Doublet2','Doublet3','Doublet4','Doublet5','lowQual']

In [ ]:
ann_filter = ann[~ann['cellType'].isin(exclude_values)]
ann_filter

In [ ]:
ann_filter = ann_filter.dropna(subset=['cellType']) # Remove missing values
ann_filter

In [ ]:
ann_filter.value_counts()

In [ ]:
ann_filter.to_csv('./processed/original_ann/ann_filter.csv')

## Filter cell

In [ ]:
adata = sc.read_h5ad('./processed/all_cell/rna.h5ad')
adata

In [ ]:
ann = pd.read_csv('./processed/original_ann/ann_filter.csv', index_col=0)
ann

In [ ]:
adata_filtered = adata[adata.obs_names.isin(ann.index)]
adata_filtered

In [ ]:
ann_sorted = ann.reindex(adata_filtered.obs_names.tolist())
ann_sorted

In [ ]:
adata_filtered.obs['cellType'] = ann_sorted['cellType'].tolist()

In [ ]:
adata_filtered.write_h5ad('./processed/ann_cell/ann_cell_rna.h5ad', compression='gzip')

In [ ]:
rna = sc.read_h5ad('./processed/ann_cell/ann_cell_atac.h5ad')
rna

In [ ]:
sc.pp.calculate_qc_metrics(rna,inplace=True)

In [ ]:
# rna.var["mt"] = rna.var_names.str.startswith("MT-")
# sc.pp.calculate_qc_metrics(
#     rna, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
# )

In [ ]:
rna

In [ ]:
sc.pl.violin(
    rna,
    ["total_counts"],
    size=0,
    multi_panel=True,
)

In [ ]:
a = pd.DataFrame(rna.obs['total_counts'].value_counts())

In [ ]:
a.to_csv('./a.csv')

## cellhint

In [ ]:
adata = sc.read_h5ad('./processed/ann_cell/ann_cell_rna.h5ad')
adata

In [ ]:
adata.obs.dataset.value_counts()

In [ ]:
sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, batch_key = 'dataset', subset = True)
sc.pp.scale(adata, max_value = 10)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
adata.write_h5ad('./processed/ann_cell/ann_cell_rna_raw_umap.h5ad', compression='gzip')

In [ ]:
sc.pl.umap(adata, color = 'dataset')

In [ ]:
sc.pl.umap(adata, color = 'cellType')

In [ ]:
import cellhint

In [ ]:
alignment = cellhint.harmonize(adata, 'dataset', 'cellType')

In [ ]:
alignment

In [ ]:
alignment.write('./processed/ann_cell/alignment.pkl')

In [ ]:
alignment = cellhint.DistanceAlignment.load('./processed/ann_cell/alignment.pkl')

In [ ]:
alignment.relation.head(10)

In [ ]:
alignment.groups

In [ ]:
alignment.reannotation

In [ ]:
cellhint.treeplot(alignment, save='cellhint.pdf')

In [ ]:
cellhint.treeplot(alignment, order_dataset = True, save='cellhint_order.pdf')

In [ ]:
dist_mat = alignment.base_distance.to_meta()
dist_mat.iloc[:5, :5]

In [ ]:
member_mat = alignment.base_distance.to_meta(turn_binary = True)
member_mat.iloc[:5, :5]

In [ ]:
import seaborn as sns
g = sns.clustermap(member_mat)
g.savefig("cellhint_clustermap.pdf", format="pdf")

In [ ]:
cellhint.treeplot(alignment.relation[alignment.groups == 'Group1'])

In [ ]:
cellhint.integrate(adata, 'sample', 'low_hierarchy')

In [ ]:
sc.tl.umap(adata)

In [ ]:
adata.write_h5ad('./processed/ann_cell/ann_cell_cellhint.h5ad', compression='gzip')

In [ ]:
sc.pl.umap(adata, color = ['dataset'])

In [ ]:
sc.pl.umap(adata, color = 'high_hierarchy')

## harmony

In [ ]:
adata = sc.read_h5ad('./processed/ann_cell/ann_cell_rna.h5ad')
adata

In [ ]:
sc.pp.calculate_qc_metrics(adata,inplace=True)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata.raw = adata
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True)

In [ ]:
adata

In [ ]:
sce.pp.harmony_integrate(adata, 'sample')
adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

adata.write('rna_neighbor.h5ad', compression='gzip')

In [ ]:
adata = sc.read_h5ad('rna_neighbor.h5ad')
adata

In [ ]:
sc.tl.umap(adata)
sc.tl.leiden(adata,resolution=1)

adata.write('rna_leiden.h5ad', compression='gzip')

In [ ]:
adata = sc.read_h5ad('./processed/ann_cell/rna_leiden.h5ad')

In [ ]:
adata_c = sc.read_h5ad('./processed/ann_cell/ann_cell_celltypist.h5ad')
adata_c

In [ ]:
adata.obs['majority_voting'] = adata_c.obs['majority_voting']

In [ ]:
sc.pl.umap(adata, color='dataset')

In [ ]:
sc.pl.umap(adata, color='majority_voting')

# Filter unannotated cells using all_ann

In [ ]:
adata = sc.read_h5ad('./processed/all_cell/rna.h5ad')
adata

In [ ]:
ann = pd.read_csv('../4_cell_ann/ann_merge.csv', index_col=0)
ann

In [ ]:
adata_filtered = adata[~adata.obs_names.isin(ann.index)]
adata_filtered

In [ ]:
adata_filtered.obs['dataset'].value_counts()

In [ ]:
adata_filtered.write_h5ad('./processed/no_annotation/all_cell_rna.h5ad', compression='gzip')

## Basic filtering

In [ ]:
adata = sc.read_h5ad('./processed/no_annotation/all_cell_rna.h5ad')
adata

In [ ]:
adata.obs['dataset'].value_counts()

In [ ]:
adata.obs['sample'].value_counts()

In [ ]:
sc.pp.calculate_qc_metrics(adata,inplace=True)

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)

In [ ]:
atac_meta = pd.read_csv('./processed/no_annotation/atac_meta.csv', index_col=0)
atac_meta

In [ ]:
adata.obs['n_peaks'] = atac_meta['n_genes_by_counts'].tolist()
adata.obs['peak_counts'] = atac_meta['total_counts'].tolist()

In [ ]:
tss = pd.read_csv('../4_cell_ann/no_ann_tss_merge.csv', index_col=0)
tss

In [ ]:
adata.obs['nucleosome_signal'] = tss['nucleosome_signal'].tolist()
adata.obs['TSS.enrichment'] = tss['TSS.enrichment'].tolist()

In [ ]:
adata

In [ ]:
sc.pl.violin(
    adata,
    ["total_counts", "pct_counts_mt"],
    size=0,
    multi_panel=True,
)

In [ ]:
sc.pl.violin(
    adata,
    ['peak_counts', "nucleosome_signal", 'TSS.enrichment'],
    size=0,
    multi_panel=True,
)

In [ ]:
adata = adata[adata.obs.total_counts < 20000, :]
adata = adata[adata.obs.total_counts > 500, :]
adata

In [ ]:
adata = adata[adata.obs.pct_counts_mt < 25, :]
adata

In [ ]:
adata = adata[adata.obs.peak_counts > 500, :]
adata = adata[adata.obs.peak_counts < 30000, :]
adata

In [ ]:
adata = adata[adata.obs['TSS.enrichment'] > 1, :]
adata = adata[adata.obs.nucleosome_signal < 2, :]
adata

In [ ]:
sc.pl.violin(
    adata,
    ["total_counts", "pct_counts_mt"],
    size=0,
    multi_panel=True,
)

In [ ]:
sc.pl.violin(
    adata,
    ['peak_counts', "nucleosome_signal", 'TSS.enrichment'],
    size=0,
    multi_panel=True,
)

In [ ]:
adata.obs['dataset'].value_counts()

In [ ]:
adata.obs['sample'].value_counts()

In [ ]:
adata.write_h5ad('./processed/no_annotation/filter_cell_rna.h5ad', compression='gzip')

# Celltypist_cellhint

In [ ]:
adata = sc.read_h5ad('./processed/no_annotation/filter_cell_rna_celltypist_cellhint.h5ad')
adata

In [ ]:
sc.pl.umap(adata, color='dataset')

In [ ]:
sc.pl.umap(adata, color='sample')

In [ ]:
sc.pl.umap(adata, color='majority_voting')

In [ ]:
sc.pl.umap(adata, color='conf_score')

# ATAC modality

In [ ]:
folder_path = '../3_rna_atac_cell/data_atac/'
mtx_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.mtx')]

In [ ]:
adata_list = [sc.read_mtx(mtx_file) for mtx_file in mtx_files]

In [ ]:
folder_path = '../3_rna_atac_cell/data_atac/'
csv_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.startswith('cell')]

In [ ]:
cell_list = [pd.read_csv(csv_file, index_col=0) for csv_file in csv_files]

In [ ]:
for i in range(len(adata_list)):
    adata_list[i].var_names = cell_list[i]['cell_id'].tolist()
    adata_list[i] = adata_list[i].T

In [ ]:
combined = adata_list[0].concatenate(adata_list[1], index_unique=None)
combined = combined.concatenate(adata_list[2], index_unique=None)
combined = combined.concatenate(adata_list[3], index_unique=None)

In [ ]:
combined = combined.concatenate(adata_list[4], index_unique=None)
combined = combined.concatenate(adata_list[5], index_unique=None)

In [ ]:
adata_list

In [ ]:
405023+1031726+185682+480289+136700+316906

In [ ]:
combined

In [ ]:
peak = pd.read_csv('../3_rna_atac_cell/data_atac/peak.csv', index_col=0)
peak

In [ ]:
combined.var_names = peak['peak'].tolist()

In [ ]:
del combined.obs['batch']
combined

In [ ]:
df = pd.DataFrame([x.split('-') for x in combined.obs_names], columns=['part1', 'part2', 'part3'])

In [ ]:
combined.obs['dataset'] = df['part1'].tolist()
combined.obs['barcode'] = df['part2'].tolist()

In [ ]:
df['part4'] = df['part1'] + '-' + df['part3']

In [ ]:
combined.obs['sample'] = df['part4'].tolist()

In [ ]:
combined.var

In [ ]:
combined.write('atac.h5ad', compression='gzip')

## annotated cells

In [ ]:
ann = pd.read_csv('./processed/original_ann/ann_filter.csv', index_col=0)
ann

In [ ]:
adata_filtered = adata[adata.obs_names.isin(ann.index)]
adata_filtered

In [ ]:
ann_sorted = ann.reindex(adata_filtered.obs_names.tolist())
ann_sorted

In [ ]:
adata_filtered.obs['cellType'] = ann_sorted['cellType'].tolist()

In [ ]:
adata_filtered.write_h5ad('./processed/ann_cell/ann_cell_atac.h5ad', compression='gzip')

## unannotated cells

In [ ]:
ann = pd.read_csv('../4_cell_ann/ann_merge.csv', index_col=0)
ann

In [ ]:
adata_filtered = adata[~adata.obs_names.isin(ann.index)]
adata_filtered

In [ ]:
adata_filtered.write_h5ad('./processed/no_annotation/all_cell_atac.h5ad', compression='gzip')

## basic filtering

In [ ]:
adata = sc.read_h5ad('./processed/no_annotation/all_cell_atac.h5ad')
adata

In [ ]:
sc.pp.calculate_qc_metrics(adata,inplace=True)

In [ ]:
adata.obs.to_csv('processed/no_annotation/atac_meta.csv')

In [ ]:
adata = sc.read_h5ad('./processed/no_annotation/all_cell_atac.h5ad')
adata

In [ ]:
adata_a = sc.read_h5ad('./processed/no_annotation/filter_cell_rna.h5ad')

In [ ]:
adata_a

In [ ]:
adata_filter = adata[adata_a.obs_names.tolist(),:]

In [ ]:
adata_filter.write_h5ad('./processed/no_annotation/filter_cell_atac.h5ad', compression='gzip')

# Celltypist and Cellhint for all cell

In [ ]:
import celltypist
import cellhint
import os
import scanpy as sc

In [ ]:
os.chdir('/path/to/hemelas_project/z_others/5_matrix_merge/')

## compute

In [ ]:
adata = sc.read_h5ad('./processed/ann_cell/ann_cell_rna.h5ad')
adata

In [ ]:
sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)

In [ ]:
adata = celltypist.annotate(adata, model = 'Immune_All_Low.pkl', majority_voting = True).to_adata()

In [ ]:
adata

In [ ]:
adata.obs[['predicted_labels', 'majority_voting', 'conf_score']]

In [ ]:
adata.write_h5ad('./processed/ann_cell/ann_cell_celltypist.h5ad', compression='gzip')

In [ ]:
cellhint.integrate(adata, 'sample', 'majority_voting')

In [ ]:
sc.tl.umap(adata)

In [ ]:
adata.write_h5ad('./processed/ann_cell/ann_cell_celltypist_cellhint.h5ad', compression='gzip')

In [ ]:
sc.pl.umap(adata, color = ['dataset', 'sample'], wspace = 0.5)

In [ ]:
sc.pl.umap(adata, color = 'majority_voting')

## check

In [ ]:
adata = sc.read_h5ad('./processed/ann_cell/ann_cell_celltypist_cellhint.h5ad')
adata

In [ ]:
adata.obs

In [ ]:
sc.pl.umap(adata, color='dataset')

In [ ]:
sc.pl.umap(adata, color='majority_voting')

In [ ]:
sc.pl.umap(adata, color='predicted_labels')

In [ ]:
sc.pl.umap(adata, color='conf_score')